# EcoSort AI - Nonlinear and Complementary Feature Experiments
This notebook documents the fixed-split experiments. The historical Colab notebook and deployed model remain separate. Accuracy targets are checked against measured results, not guaranteed. All held-out images remain outside fitting.
## 1. Check the environment and fixed partitions
Use the EcoSort local Python environment. The manifest retains the original grouped train, validation and test assignments.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import pandas as pd
import torch
PROJECT = Path.cwd()
assert (PROJECT / 'train_nonlinear.py').exists()
manifest = pd.read_csv(PROJECT / 'data/expanded/split_manifest.csv')
assert manifest.sha256.is_unique
assert manifest.groupby('group_id').split.nunique().max() == 1
print(manifest.groupby('split').size())
print('Apple GPU available:', torch.backends.mps.is_available())

## 2. Train nonlinear classification and neighbor candidates
Reuse all frozen OpenCLIP training features. Compare a material-level head, a finer object-label head aggregated into the same 17 categories, and similarities to training images only. Validation selects the candidate and checkpoint. Existing completed evidence is loaded instead of overwritten.

In [ ]:
record = PROJECT / 'trained_nonlinear/training_completed.json'
if not record.exists():
    subprocess.run([sys.executable, '-u', 'train_nonlinear.py'], check=True)
print(json.loads(record.read_text()))

## 3. Extract complementary DINOv2 features
The pinned, checksum-verified DINOv2 Small encoder is frozen. Full-image 224-pixel letterboxing preserves object boundaries; positional embeddings are adapted by timm for this experimental input size. Concatenate normalized class-token and mean patch features with the existing CLIP features. Extraction visits every retained image and resumes from verified markers. No labels are used to fit the frozen encoders.

In [ ]:
subprocess.run([sys.executable, '-u', 'extract_dino_features.py'], check=True)

## 4. Fit and select the complementary-feature candidate
Train the heads using the unchanged training partition. Select weights and calibration using validation only, then evaluate the selected model on the reused public test benchmark. The completion record explicitly reports whether both accuracy values reach 95%. Repeated benchmark evaluation is not a substitute for new independent local validation.

In [ ]:
record = PROJECT / 'trained_multiview/training_completed.json'
if not record.exists():
    subprocess.run([sys.executable, '-u', 'train_nonlinear.py', '--features', 'features_clip_dino', '--output', 'trained_multiview'], check=True)
print(json.loads(record.read_text()))

## 5. Evaluate real-photo regressions separately
The six user photos, thirty web photos, TACO crops and original ten photos are previously observed diagnostics. Never fit on them or call them a fresh independent test. Local labels include owner confirmation for the board and plastic puzzle piece; the ceramic mug and kitchen towel retain visual assumptions. Review flags do not remove errors from the denominator.

In [ ]:
for name in ['trained_nonlinear', 'trained_multiview']:
    results = PROJECT / name
    record = results / 'external_diagnostics.json'
    if (results / 'training_completed.json').exists():
        if not record.exists():
            subprocess.run([sys.executable, '-u', 'evaluate_candidate.py', name], check=True)
        print(name, json.loads(record.read_text()))

## 6. Check deployment and report evidence
A candidate is not automatically deployed merely because its public benchmark score increases. Read the active pointer and keep public accuracy, real-photo diagnostics and interface functionality separate. The review documents describe implemented functions, actual development problems and outstanding work without modifying the team's PDF.

In [ ]:
print('Active website model:', json.loads((PROJECT / 'active_model.json').read_text()))
print('Website: http://127.0.0.1:7861/')
print('Chinese report review:', PROJECT / 'FINAL_REPORT_REVIEW_ZH.md')
print('English report review:', PROJECT / 'FINAL_REPORT_REVIEW_EN.md')